# الدرس 10 - وكلاء الذكاء الاصطناعي في الإنتاج

في هذا الدرس ستتعلم **أنماط الإنتاج** لوكلاء الذكاء الاصطناعي باستخدام إطار وكلاء مايكروسوفت (Python). نغطي:

- **القابلية للملاحظة** — إضافة تتبع الزمن وتسجيل السجلات لتفاعلات الوكلاء
- **التقييم** — استخدام وكيل مقيم لتقييم جودة الاستجابات
- **إدارة التكلفة** — استراتيجيات لتحسين استخدام التوكنات واختيار النماذج

السيناريو هو a **وكيل سفر** يساعد المستخدمين على تخطيط الرحلات، مع وضع طبقات للمراقبة والتقييم فوقه.


## الإعداد


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## اعتبارات الإنتاج

يتطلب نقل وكلاء الذكاء الاصطناعي من النماذج الأولية إلى مرحلة الإنتاج الانتباه الدقيق لثلاثة ركائز:

1. **القابلية للملاحظة** — تحتاج إلى رؤية لما يفعله الوكيل، وكم من الوقت يستغرق، وأي الأدوات يستدعيها. بدون التتبع وتسجيل السجلات، يصبح استكشاف أخطاء الإنتاج وإصلاحها شبه مستحيل.

2. **التقييم** — تضمن عمليات التحقق الآلية للجودة بقاء ردود الوكيل دقيقة وكاملة ومفيدة مع مرور الوقت. يمكن لوكيل مُقَيِّم أن يقيم الاستجابات وفقًا لمعايير محددة.

3. **إدارة التكاليف** — يؤثر استخدام التوكنات مباشرة على التكلفة. تساعد استراتيجيات مثل تحسين المطالبات، اختيار النموذج، والتخزين المؤقت في إبقاء النفقات تحت السيطرة دون التضحية بالجودة.


## بناء وكيل قابل للمراقبة

نُعرّف أدوات السفر ونغلف استدعاء الوكيل بقياس الزمن حتى نتمكن من مراقبة الكمون. في بيئة الإنتاج ستقوم بدمج ذلك مع OpenTelemetry أو نظام تتبّع مشابه.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## أنماط التقييم

نمط إنتاج شائع هو استخدام وكيل ثانٍ كـ **المقيّم**. يقيم المقيّم استجابة الوكيل الأساسي مقابل معايير محددة مسبقًا مثل الاكتمال والدقة والفائدة.

هذا يتيح:
- بوابات جودة آلية قبل وصول الردود إلى المستخدمين
- الكشف عن التراجعات عندما تتغير المطالبات أو النماذج
- المراقبة المستمرة لأداء الوكيل بمرور الوقت


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## استراتيجيات إدارة التكلفة

التحكم في التكاليف أمر حاسم لوكلاء الذكاء الاصطناعي في بيئات الإنتاج. فيما يلي استراتيجيات رئيسية:

| الاستراتيجية | الوصف |
|---|---|
| **تحسين الموجه** | اجعل تعليمات النظام موجزة. قم بإزالة السياق الزائد لتقليل عدد الرموز في الإدخال. |
| **اختيار النموذج** | استخدم نماذج أصغر وأرخص (على سبيل المثال GPT-4o-mini) للمهام البسيطة مثل التصنيف أو الاستخراج، واحتفظ بالنماذج الأكبر للمهام التي تتطلب استدلالًا معقدًا. |
| **التخزين المؤقت** | خزن نتائج الأدوات والاستعلامات المتكررة لتجنب استدعاءات API المكررة. |
| **ميزانيات الرموز** | حدد حدود `max_tokens` لمنع الحصول على استجابات طويلة بشكل غير متوقع. |
| **التجميع** | قم بتجميع استفسارات المستخدم المتعددة في استدعاء API واحد حيثما أمكن. |

عمليًا، النهج متعدد المستويات يعمل جيدًا: قم بتوجيه الطلبات البسيطة إلى نموذج سريع وغير مكلف وتصعيد الاستفسارات المعقدة فقط إلى نموذج أكثر قدرة.


## الملخص

في هذا الدرس تعلّمت كيفية:

1. **إضافة إمكانية المراقبة** لتفاعلات الوكيل باستخدام التوقيت والتسجيل، مما يمهد الأساس للتتبّع والمراقبة.
2. **تقييم استجابات الوكيل** تلقائيًا باستخدام وكيل مقيم يمنح درجات للاكتمال والدقة والفائدة.
3. **إدارة التكاليف** من خلال تحسين المطالبات، اختيار النموذج، التخزين المؤقت، وميزانيات الرموز.

تساعد أنماط الإنتاج هذه في ضمان أن وكلاء الذكاء الاصطناعي لديك موثوقون، قابلة للقياس، وفعّالون من حيث التكلفة على نطاق واسع.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
إخلاء المسؤولية:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية Co-op Translator (https://github.com/Azure/co-op-translator). بينما نسعى جاهدين لتحقيق الدقة، يُرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر المعتمد. للمعلومات الحرجة، يوصى بالاستعانة بترجمة بشرية مهنية. لا نتحمل أي مسؤولية عن أي سوء فهم أو تفسيرات خاطئة تنشأ عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
